# Syllable Onset And Transition Example Videos Across Brain Regions

This notebook is a lightweight viewer for pulling matched example clips across `NAc` and `mPFC`.

It supports both:
- `syllable_onset`
- `transition_onset`

The notebook reuses the exported alignment outputs plus the per-frame KPMS table so you can:
- rank events by count or DA metrics
- inspect matched examples across regions
- render GIFs with real video when available, or pose-only fallback when not


In [ ]:
from pathlib import Path
import math
import re
import json

import cv2
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import animation
from IPython.display import HTML, display

sns.set_context('talk')
sns.set_style('whitegrid')

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'Keypoint_moseq_analysis':
    ANALYSIS_DIR = NOTEBOOK_DIR
    REPO_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / 'Keypoint_moseq_analysis').exists():
    REPO_ROOT = NOTEBOOK_DIR
    ANALYSIS_DIR = NOTEBOOK_DIR / 'Keypoint_moseq_analysis'
else:
    ANALYSIS_DIR = NOTEBOOK_DIR
    REPO_ROOT = NOTEBOOK_DIR.parent

OUTPUT_DIR = ANALYSIS_DIR / 'Pose_Tracking' / 'syllable_transition_da_alignment_outputs'
EVENT_COMPARE_DIR = OUTPUT_DIR / 'event_definition_comparison_outputs'

FRAMES_PATH = OUTPUT_DIR / 'subject_syllable_frames_with_bouts.csv'
TRANSITIONS_PATH = OUTPUT_DIR / 'subject_syllable_transitions.csv'
ALIGNED_SYLLABLE_ONSETS_PATH = EVENT_COMPARE_DIR / 'aligned_syllable_onsets.csv'
ALIGNED_TRANSITION_ONSETS_PATH = EVENT_COMPARE_DIR / 'aligned_transition_onsets.csv'

VIDEO_SEARCH_ROOTS = [
    Path(r'C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage\SLEAP\HC_Videos_Only'),
    Path(r'C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage\SLEAP\all_tdt'),
    Path(r'C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage'),
]

REGION_ORDER = ['NAc', 'mPFC']
BOUT_PHASES = ['Short_Term-1', 'Short_Term-2', 'Long_Term-1', 'Novel-1']
DEFAULT_WINDOW_S = 1.5
DEFAULT_N_EXAMPLES = 4
DEFAULT_SOCIAL_ONLY = True
TRACK_COLORS_RGB = {'subject': (21, 97, 111), 'agent': (196, 69, 54)}
TRACK_COLORS_HEX = {'subject': '#15616F', 'agent': '#C44536'}


In [ ]:
def _normalize_flag(series_or_values):
    series = pd.Series(series_or_values)
    mapped = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            'true': True,
            'false': False,
            '1': True,
            '0': False,
            'yes': True,
            'no': False,
            'nan': False,
            'none': False,
            '': False,
        })
    )
    return mapped.fillna(False)


def _decode_if_bytes(values):
    out = []
    for value in values:
        out.append(value.decode() if isinstance(value, (bytes, np.bytes_)) else str(value))
    return out


def _extract_edge_pairs_from_h5(h5_file, n_nodes):
    for key in ['edge_inds', 'edge_ind', 'edge_index', 'skeleton_inds']:
        if key in h5_file:
            arr = np.asarray(h5_file[key])
            if arr.ndim == 2 and 2 in arr.shape:
                if arr.shape[0] == 2 and arr.shape[1] != 2:
                    arr = arr.T
                pairs = [(int(a), int(b)) for a, b in arr[:, :2] if 0 <= int(a) < n_nodes and 0 <= int(b) < n_nodes]
                if pairs:
                    return pairs
    return [(i, i + 1) for i in range(max(0, n_nodes - 1))]


def _load_pose_bundle(source_h5):
    with h5py.File(source_h5, 'r') as f:
        track_names = _decode_if_bytes(f['track_names'][:])
        node_names = _decode_if_bytes(f['node_names'][:])
        coords = np.asarray(f['tracks'][:]).transpose((3, 2, 1, 0))
        edge_pairs = _extract_edge_pairs_from_h5(f, n_nodes=len(node_names))
    return {
        'coords': coords,
        'track_names': track_names,
        'node_names': node_names,
        'edge_pairs': edge_pairs,
    }


def _candidate_video_stems(source_h5):
    source_h5 = Path(source_h5)
    stem = source_h5.stem
    if stem.endswith('.analysis'):
        stem = Path(stem).stem
    base_candidates = [
        stem,
        stem.replace('_converted', ''),
        stem.replace('_converted', '') + '_converted',
    ]
    match = re.search(r'(\d+_\d+_\d+_Home_Cage_.*)', stem)
    if match:
        trimmed = match.group(1)
        base_candidates.extend([
            trimmed,
            trimmed.replace('_converted', ''),
            trimmed.replace('_converted', '') + '_converted',
        ])
    ordered = []
    seen = set()
    for item in base_candidates:
        if item not in seen:
            ordered.append(item)
            seen.add(item)
    return ordered


def _extract_subject_tokens(*values):
    tokens = set()
    for value in values:
        if value is None:
            continue
        text = str(value).lower()
        tokens.update(re.findall(r'[np]{1,2}\d+', text))
    return tokens


def _path_matches_subject(video_path, subject_tokens):
    if not subject_tokens:
        return True
    name = Path(video_path).stem.lower()
    path_tokens = set(re.findall(r'[np]{1,2}\d+', name))
    if path_tokens & subject_tokens:
        return True
    normalized_name = re.sub(r'[^a-z0-9]+', '_', name)
    for token in subject_tokens:
        if re.search(rf'(^|_)' + re.escape(token) + r'(_|$)', normalized_name):
            return True
    return False


def _score_video_candidate(video_path, stems, subject_tokens):
    stem_name = Path(video_path).stem.lower()
    score = 0
    if any(stem_name == stem.lower() for stem in stems):
        score += 4
    if any(stem.lower() in stem_name for stem in stems):
        score += 2
    if _path_matches_subject(video_path, subject_tokens):
        score += 8
    return score


def _find_video_for_source_h5(source_h5, subject_name=None, recording_name=None):
    stems = _candidate_video_stems(source_h5)
    exts = ['.mp4', '.avi', '.mov', '.mkv']
    subject_tokens = _extract_subject_tokens(subject_name, recording_name, source_h5)
    best_match = None
    best_score = -1
    for root in VIDEO_SEARCH_ROOTS:
        if not root.exists():
            continue
        for stem in stems:
            for ext in exts:
                candidates = []
                candidates.extend(root.rglob(stem + ext))
                candidates.extend(root.rglob('*' + stem + ext))
                seen = set()
                deduped = []
                for candidate in candidates:
                    key = str(candidate).lower()
                    if key in seen:
                        continue
                    seen.add(key)
                    deduped.append(candidate)
                for candidate in deduped:
                    if not _path_matches_subject(candidate, subject_tokens):
                        continue
                    score = _score_video_candidate(candidate, stems, subject_tokens)
                    if score > best_score:
                        best_match = candidate
                        best_score = score
    return best_match


def _read_video_clip(video_path, frame_start, frame_stop):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')
    frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_start))
    for _ in range(int(frame_start), int(frame_stop) + 1):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def _overlay_pose_on_frame(frame_rgb, pts, edge_pairs, color_rgb):
    out = frame_rgb.copy()
    color = tuple(int(c) for c in color_rgb)
    for a, b in edge_pairs:
        pa = pts[a]
        pb = pts[b]
        if np.any(np.isnan(pa)) or np.any(np.isnan(pb)):
            continue
        cv2.line(out, tuple(np.round(pa).astype(int)), tuple(np.round(pb).astype(int)), color, 2)
    for point in pts:
        if np.any(np.isnan(point)):
            continue
        cv2.circle(out, tuple(np.round(point).astype(int)), 4, color, -1)
    return out


In [ ]:
frames_df = pd.read_csv(FRAMES_PATH)
transitions_df = pd.read_csv(TRANSITIONS_PATH)
aligned_syllable_onsets = pd.read_csv(ALIGNED_SYLLABLE_ONSETS_PATH)
aligned_transition_onsets = pd.read_csv(ALIGNED_TRANSITION_ONSETS_PATH)

frames_df['frame_index'] = pd.to_numeric(frames_df['frame_index'], errors='coerce').astype(int)
frames_df['time_s'] = pd.to_numeric(frames_df['time_s'], errors='coerce')
frames_df['fps'] = pd.to_numeric(frames_df['fps'], errors='coerce')
frames_df['syllable'] = pd.to_numeric(frames_df['syllable'], errors='coerce').astype(int)
frames_df['social_proximity'] = _normalize_flag(frames_df.get('social_proximity', False))

transitions_df['entry_time_s'] = pd.to_numeric(transitions_df['entry_time_s'], errors='coerce')
transitions_df['entry_frame_index'] = pd.to_numeric(transitions_df['entry_frame_index'], errors='coerce').astype(int)
transitions_df['from_syllable'] = pd.to_numeric(transitions_df['from_syllable'], errors='coerce').astype(int)
transitions_df['to_syllable'] = pd.to_numeric(transitions_df['to_syllable'], errors='coerce').astype(int)
transitions_df['social_proximity'] = _normalize_flag(transitions_df.get('social_proximity', False))
transitions_df['changed'] = _normalize_flag(transitions_df.get('changed', True))

aligned_syllable_onsets['event_time_s'] = pd.to_numeric(aligned_syllable_onsets['event_time_s'], errors='coerce')
aligned_syllable_onsets['syllable'] = pd.to_numeric(aligned_syllable_onsets['syllable'], errors='coerce').astype(int)
aligned_syllable_onsets['social_proximity_at_event'] = _normalize_flag(aligned_syllable_onsets.get('social_proximity_at_event', False))

aligned_transition_onsets['event_time_s'] = pd.to_numeric(aligned_transition_onsets['event_time_s'], errors='coerce')
aligned_transition_onsets['entry_frame_index'] = pd.to_numeric(aligned_transition_onsets['entry_frame_index'], errors='coerce').astype(int)
aligned_transition_onsets['from_syllable'] = pd.to_numeric(aligned_transition_onsets['from_syllable'], errors='coerce').astype(int)
aligned_transition_onsets['to_syllable'] = pd.to_numeric(aligned_transition_onsets['to_syllable'], errors='coerce').astype(int)
aligned_transition_onsets['social_proximity_at_event'] = _normalize_flag(aligned_transition_onsets.get('social_proximity_at_event', False))

recording_info = (
    frames_df.sort_values(['recording_name', 'frame_index'])
    .groupby('recording_name', as_index=False)
    .first()[['recording_name', 'subject_name', 'brain_region', 'source_h5', 'fps']]
)

frames_subject = frames_df[frames_df['track_name'] == 'subject'].copy().sort_values(['recording_name', 'frame_index']).reset_index(drop=True)
frame_break = (
    frames_subject['recording_name'].ne(frames_subject['recording_name'].shift())
    | frames_subject['syllable'].ne(frames_subject['syllable'].shift())
    | (frames_subject['frame_index'] != frames_subject['frame_index'].shift().fillna(-1) + 1)
)
frames_subject['run_id'] = frame_break.cumsum()

run_events = (
    frames_subject.groupby('run_id', as_index=False)
    .agg(
        recording_name=('recording_name', 'first'),
        subject_name=('subject_name', 'first'),
        brain_region=('brain_region', 'first'),
        source_h5=('source_h5', 'first'),
        fps=('fps', 'first'),
        bout_phase=('bout_phase', 'first'),
        syllable=('syllable', 'first'),
        start_frame_index=('frame_index', 'min'),
        end_frame_index=('frame_index', 'max'),
        start_time_s=('time_s', 'min'),
        end_time_s=('time_s', 'max'),
        social_proximity=('social_proximity', 'max'),
        social_role=('social_role', 'first'),
        behavior_active=('behavior_active', 'first'),
    )
)
run_events['duration_s'] = run_events['end_time_s'] - run_events['start_time_s'] + (1.0 / run_events['fps'].replace(0, np.nan))
run_events['syllable_label'] = run_events['syllable'].astype(str)
run_events['event_kind'] = 'syllable_onset'
run_events['event_label'] = run_events['syllable_label']
run_events['event_time_s_round'] = run_events['start_time_s'].round(4)

aligned_syllable_onsets['event_time_s_round'] = aligned_syllable_onsets['event_time_s'].round(4)
syllable_onset_events = run_events.merge(
    aligned_syllable_onsets.drop(columns=['subject_name', 'brain_region'], errors='ignore'),
    left_on=['recording_name', 'syllable', 'event_time_s_round'],
    right_on=['recording_name', 'syllable', 'event_time_s_round'],
    how='left',
    suffixes=('', '_aligned'),
)
syllable_onset_events['event_time_s'] = syllable_onset_events['start_time_s']
syllable_onset_events['event_kind'] = 'syllable_onset'
syllable_onset_events['event_label'] = syllable_onset_events['syllable'].astype(str)
syllable_onset_events['social_proximity_event'] = _normalize_flag(
    syllable_onset_events.get('social_proximity_at_event', syllable_onset_events.get('social_proximity', False))
)

transition_onset_events = aligned_transition_onsets.merge(
    recording_info[['recording_name', 'source_h5', 'fps']],
    on='recording_name',
    how='left',
)
transition_onset_events['event_kind'] = 'transition_onset'
transition_onset_events['event_label'] = transition_onset_events['transition_label'].astype(str)
transition_onset_events['social_proximity_event'] = _normalize_flag(
    transition_onset_events.get('social_proximity_at_event', transition_onset_events.get('social_proximity', False))
)

print(f'Loaded {len(syllable_onset_events):,} syllable onsets')
print(f'Loaded {len(transition_onset_events):,} transition onsets')
display(recording_info.head())
display(syllable_onset_events[['recording_name', 'brain_region', 'syllable', 'start_time_s', 'start_frame_index', 'delta_mean']].head())
display(transition_onset_events[['recording_name', 'brain_region', 'transition_label', 'event_time_s', 'entry_frame_index', 'delta_mean']].head())


In [ ]:
def get_event_table(event_kind='syllable_onset'):
    if event_kind == 'syllable_onset':
        return syllable_onset_events.copy()
    if event_kind == 'transition_onset':
        return transition_onset_events.copy()
    raise ValueError("event_kind must be 'syllable_onset' or 'transition_onset'")


def coerce_event_label(event_kind, label):
    if event_kind == 'syllable_onset':
        return str(int(float(label)))
    if event_kind == 'transition_onset':
        if isinstance(label, str):
            return label.replace(' ', '')
        if isinstance(label, (tuple, list)) and len(label) == 2:
            return f'{int(label[0])}->{int(label[1])}'
    raise ValueError('Label format not understood for this event_kind')


def rank_event_labels(event_kind='syllable_onset', metric='n_events', region=None, bout_phase=None, social_only=None, min_events=1):
    df = get_event_table(event_kind)
    if region is not None:
        df = df[df['brain_region'] == region].copy()
    if bout_phase is not None:
        bout_series = df['bout_phase'] if 'bout_phase' in df.columns else pd.Series(index=df.index, dtype=object)
        bout_context = df['bout_phase_context'] if 'bout_phase_context' in df.columns else pd.Series(index=df.index, dtype=object)
        df = df[bout_series.eq(bout_phase) | bout_context.eq(bout_phase)].copy()
    if social_only is True:
        df = df[df['social_proximity_event']].copy()

    summary = (
        df.groupby(['brain_region', 'event_label'], as_index=False)
        .agg(
            n_events=('event_label', 'size'),
            n_recordings=('recording_name', 'nunique'),
            n_subjects=('subject_name', 'nunique'),
            mean_delta_mean=('delta_mean', 'mean'),
            mean_peak_post=('peak_post', 'mean'),
            mean_z_at_event=('zscore_at_event', 'mean'),
        )
    )
    summary = summary[summary['n_events'] >= min_events].copy()
    sort_col = {
        'n_events': 'n_events',
        'delta_mean': 'mean_delta_mean',
        'peak_post': 'mean_peak_post',
        'zscore_at_event': 'mean_z_at_event',
    }.get(metric, metric)
    return summary.sort_values(['brain_region', sort_col], ascending=[True, False]).reset_index(drop=True)


def summarize_label_across_regions(event_kind, label, social_only=None):
    label = coerce_event_label(event_kind, label)
    df = get_event_table(event_kind)
    df = df[df['event_label'] == label].copy()
    if social_only is True:
        df = df[df['social_proximity_event']].copy()
    out = (
        df.groupby('brain_region', as_index=False)
        .agg(
            n_events=('event_label', 'size'),
            n_recordings=('recording_name', 'nunique'),
            n_subjects=('subject_name', 'nunique'),
            mean_delta_mean=('delta_mean', 'mean'),
            mean_peak_post=('peak_post', 'mean'),
            mean_z_at_event=('zscore_at_event', 'mean'),
        )
        .sort_values('brain_region')
        .reset_index(drop=True)
    )
    return out


rank_event_labels('syllable_onset', metric='n_events', social_only=None).head(10)


In [ ]:
def sample_event_examples(event_kind='syllable_onset', label=None, region=None, bout_phase=None, n_examples=4, random_state=0, social_only=None, one_per_region=False, one_per_bout_phase=False):
    df = get_event_table(event_kind)
    if label is not None:
        label = coerce_event_label(event_kind, label)
        df = df[df['event_label'] == label].copy()
    if region is not None:
        regions = [region] if isinstance(region, str) else list(region)
        df = df[df['brain_region'].isin(regions)].copy()
    if bout_phase is not None:
        bout_series = df['bout_phase'] if 'bout_phase' in df.columns else pd.Series(index=df.index, dtype=object)
        bout_context = df['bout_phase_context'] if 'bout_phase_context' in df.columns else pd.Series(index=df.index, dtype=object)
        df = df[bout_series.eq(bout_phase) | bout_context.eq(bout_phase)].copy()
    if social_only is True:
        df = df[df['social_proximity_event']].copy()
    if df.empty:
        return df

    if one_per_region:
        pieces = []
        for idx, region_name in enumerate(REGION_ORDER):
            part = df[df['brain_region'] == region_name].copy()
            if part.empty:
                continue
            pieces.append(part.sample(n=1, random_state=random_state + idx))
        if pieces:
            return pd.concat(pieces, ignore_index=True).sort_values(['brain_region', 'subject_name', 'event_time_s']).reset_index(drop=True)
        return df.iloc[0:0].copy()

    if one_per_bout_phase:
        pieces = []
        available = [phase for phase in BOUT_PHASES if phase in df['bout_phase'].dropna().unique()]
        for idx, phase in enumerate(available):
            part = df[df['bout_phase'] == phase].copy()
            if part.empty:
                continue
            pieces.append(part.sample(n=1, random_state=random_state + idx))
        if pieces:
            return pd.concat(pieces, ignore_index=True).sort_values(['bout_phase', 'brain_region', 'event_time_s']).reset_index(drop=True)
        return df.iloc[0:0].copy()

    n_take = min(n_examples, len(df))
    return df.sample(n=n_take, random_state=random_state).sort_values(['brain_region', 'subject_name', 'event_time_s']).reset_index(drop=True)


def _get_center_frame_index(event):
    if 'entry_frame_index' in event.index and pd.notna(event['entry_frame_index']):
        return int(event['entry_frame_index'])
    if 'start_frame_index' in event.index and pd.notna(event['start_frame_index']):
        return int(event['start_frame_index'])
    raise ValueError('No frame index found for event')


def _get_event_time_s(event):
    for col in ['event_time_s', 'start_time_s', 'entry_time_s']:
        if col in event.index and pd.notna(event[col]):
            return float(event[col])
    return np.nan


def _format_event_label_for_file(label):
    return str(label).replace('->', '_to_').replace('/', '_').replace(' ', '')


def _format_event_title(event_kind, event):
    if event_kind == 'syllable_onset':
        core = f"syllable {event['event_label']}"
    else:
        core = f"transition {event['event_label']}"
    return (
        f"{event['subject_name']} | {event['brain_region']} | {event.get('bout_phase', 'NA')}\n"
        f"{core} | social={bool(event.get('social_proximity_event', False))}"
    )


In [ ]:
def _render_event_overlay_frame(event_kind, event, rel_time_s=0.0, use_actual_video=True):
    source_h5 = Path(event['source_h5'])
    fps = float(event['fps']) if pd.notna(event['fps']) else 10.0
    pose_bundle = _load_pose_bundle(source_h5)
    center_idx = _get_center_frame_index(event)
    target_idx = int(np.clip(round(center_idx + rel_time_s * fps), 0, pose_bundle['coords'].shape[0] - 1))
    pose_frame = pose_bundle['coords'][target_idx]

    video_path = _find_video_for_source_h5(
        source_h5,
        subject_name=event.get('subject_name'),
        recording_name=event.get('recording_name'),
    ) if use_actual_video else None
    using_actual_video = video_path is not None

    if using_actual_video:
        video_frames = _read_video_clip(video_path, target_idx, target_idx)
        if not video_frames:
            using_actual_video = False
            canvas = None
        else:
            canvas = video_frames[0].copy()
    else:
        canvas = None

    if canvas is None:
        flat = pose_frame.transpose(2, 0, 1).reshape(-1, 2)
        valid = flat[~np.isnan(flat).any(axis=1)]
        if len(valid) == 0:
            return None
        xy_min = np.floor(valid.min(axis=0) - 20).astype(int)
        xy_max = np.ceil(valid.max(axis=0) + 20).astype(int)
        width = int(max(10, xy_max[0] - xy_min[0] + 1))
        height = int(max(10, xy_max[1] - xy_min[1] + 1))
        canvas = np.ones((height, width, 3), dtype=np.uint8) * 255
        pose_frame = pose_frame.copy()
        pose_frame[0, :, :] = pose_frame[0, :, :] - xy_min[0]
        pose_frame[1, :, :] = pose_frame[1, :, :] - xy_min[1]

    for track_name in ['subject', 'agent']:
        if track_name not in pose_bundle['track_names']:
            continue
        track_idx = pose_bundle['track_names'].index(track_name)
        pts = pose_frame[:, :, track_idx]
        canvas = _overlay_pose_on_frame(canvas, pts, pose_bundle['edge_pairs'], TRACK_COLORS_RGB.get(track_name, (128, 128, 128)))

    return {
        'image': canvas,
        'fps': fps,
        'target_idx': target_idx,
        'using_actual_video': using_actual_video,
        'video_path': video_path,
    }


def make_event_example_videos(event_kind='syllable_onset', label=None, region=None, bout_phase=None, n_examples=4, window_s=DEFAULT_WINDOW_S, save_dir=None, random_state=0, social_only=None, use_actual_video=True, one_per_region=False, one_per_bout_phase=False):
    examples = sample_event_examples(
        event_kind=event_kind,
        label=label,
        region=region,
        bout_phase=bout_phase,
        n_examples=n_examples,
        random_state=random_state,
        social_only=social_only,
        one_per_region=one_per_region,
        one_per_bout_phase=one_per_bout_phase,
    )
    if examples.empty:
        print('No matching events found.')
        return examples

    save_path = None if save_dir is None else Path(save_dir)
    if save_path is not None:
        save_path.mkdir(parents=True, exist_ok=True)

    for _, event in examples.iterrows():
        source_h5 = Path(event['source_h5'])
        fps = float(event['fps']) if pd.notna(event['fps']) else 10.0
        pose_bundle = _load_pose_bundle(source_h5)
        center_idx = _get_center_frame_index(event)
        half_window_frames = int(round(window_s * fps))
        frame_start = max(0, center_idx - half_window_frames)
        frame_stop = min(pose_bundle['coords'].shape[0] - 1, center_idx + half_window_frames)
        center_in_clip = center_idx - frame_start
        pose_clip = pose_bundle['coords'][frame_start:frame_stop + 1]

        video_path = _find_video_for_source_h5(
            source_h5,
            subject_name=event.get('subject_name'),
            recording_name=event.get('recording_name'),
        ) if use_actual_video else None
        using_actual_video = video_path is not None
        if using_actual_video:
            try:
                background_frames = _read_video_clip(video_path, frame_start, frame_stop)
                if len(background_frames) != len(pose_clip):
                    using_actual_video = False
                    background_frames = []
            except Exception as exc:
                print(f'Video read failed for {video_path}: {exc}')
                using_actual_video = False
                background_frames = []
        else:
            background_frames = []

        if not using_actual_video:
            flat = pose_clip.transpose(0, 3, 1, 2).reshape(-1, 2)
            valid = flat[~np.isnan(flat).any(axis=1)]
            if len(valid) == 0:
                continue
            xy_min = np.floor(valid.min(axis=0) - 20).astype(int)
            xy_max = np.ceil(valid.max(axis=0) + 20).astype(int)
            width = int(max(10, xy_max[0] - xy_min[0] + 1))
            height = int(max(10, xy_max[1] - xy_min[1] + 1))
            background_frames = [np.ones((height, width, 3), dtype=np.uint8) * 255 for _ in range(len(pose_clip))]
            pose_clip = pose_clip.copy()
            pose_clip[:, 0, :, :] = pose_clip[:, 0, :, :] - xy_min[0]
            pose_clip[:, 1, :, :] = pose_clip[:, 1, :, :] - xy_min[1]

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.axis('off')
        image_artist = ax.imshow(background_frames[0])
        title = ax.set_title('')

        def update(i):
            canvas = background_frames[i].copy()
            for track_name in ['subject', 'agent']:
                if track_name not in pose_bundle['track_names']:
                    continue
                track_idx = pose_bundle['track_names'].index(track_name)
                pts = pose_clip[i, :, :, track_idx]
                canvas = _overlay_pose_on_frame(canvas, pts, pose_bundle['edge_pairs'], TRACK_COLORS_RGB.get(track_name, (128, 128, 128)))
            image_artist.set_data(canvas)
            rel_t = (i - center_in_clip) / fps
            title.set_text(_format_event_title(event_kind, event) + f" | t={rel_t:+.2f}s | {'video' if using_actual_video else 'pose-only'}")
            return image_artist, title

        ani = animation.FuncAnimation(fig, update, frames=len(background_frames), interval=1000 / fps, blit=False)
        display(HTML(ani.to_jshtml()))

        if save_path is not None:
            label_slug = _format_event_label_for_file(event['event_label'])
            event_time_tenths = int(round(_get_event_time_s(event) * 10))
            gif_name = f"{event_kind}_{event['subject_name']}_{event['brain_region']}_{event.get('bout_phase', 'NA')}_{label_slug}_{event_time_tenths}.gif"
            ani.save(save_path / gif_name, writer=animation.PillowWriter(fps=max(1, int(round(fps)))))

        plt.close(fig)

    return examples


In [ ]:
def plot_event_comparison_across_regions(event_kind='syllable_onset', label=None, bout_phase=None, social_only=None, rel_time_s=0.0, use_actual_video=True, max_examples_per_region=3, random_state=0, ncols=3):
    label = coerce_event_label(event_kind, label)
    matches = get_event_table(event_kind)
    matches = matches[matches['event_label'] == label].copy()
    if bout_phase is not None:
        bout_series = matches['bout_phase'] if 'bout_phase' in matches.columns else pd.Series(index=matches.index, dtype=object)
        bout_context = matches['bout_phase_context'] if 'bout_phase_context' in matches.columns else pd.Series(index=matches.index, dtype=object)
        matches = matches[bout_series.eq(bout_phase) | bout_context.eq(bout_phase)].copy()
    if social_only is True:
        matches = matches[matches['social_proximity_event']].copy()
    if matches.empty:
        print('No matching events found.')
        return matches

    rendered = []
    for region_name in REGION_ORDER:
        region_matches = matches[matches['brain_region'] == region_name].copy()
        if region_matches.empty:
            continue
        region_matches = region_matches.sort_values(['subject_name', 'recording_name', 'event_time_s']).drop_duplicates(subset=['recording_name'], keep='first')
        if max_examples_per_region is not None and len(region_matches) > max_examples_per_region:
            region_matches = region_matches.sample(n=max_examples_per_region, random_state=random_state).sort_values(['subject_name', 'event_time_s'])
        for _, event in region_matches.iterrows():
            frame_info = _render_event_overlay_frame(event_kind, event, rel_time_s=rel_time_s, use_actual_video=use_actual_video)
            if frame_info is not None:
                rendered.append((event, frame_info))

    if not rendered:
        print('No renderable examples found.')
        return matches

    ncols = max(1, int(ncols))
    nrows = math.ceil(len(rendered) / ncols)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (event, frame_info) in zip(axes, rendered):
        ax.imshow(frame_info['image'])
        ax.set_title(_format_event_title(event_kind, event) + f"\nt={rel_time_s:+.2f}s")
        ax.axis('off')

    for ax in axes[len(rendered):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    return pd.DataFrame([event for event, _ in rendered]).reset_index(drop=True)


## Quick Ranking Tables

Use these to find labels worth inspecting before rendering clips.

In [ ]:
display(rank_event_labels('syllable_onset', metric='n_events', social_only=None).head(20))
display(rank_event_labels('transition_onset', metric='n_events', social_only=True).head(20))

## Compare One Label Across Regions

For syllables, pass the syllable number. For transitions, pass either `'13->5'` or `(13, 5)`.

In [ ]:
summarize_label_across_regions('syllable_onset', 1, social_only=None)


In [ ]:
plot_event_comparison_across_regions(
    event_kind='syllable_onset',
    label=1,
    social_only=None,
    rel_time_s=0.0,
    use_actual_video=True,
    max_examples_per_region=3,
)


In [ ]:
summarize_label_across_regions('transition_onset', '13->5', social_only=True)


In [ ]:
plot_event_comparison_across_regions(
    event_kind='transition_onset',
    label='13->5',
    social_only=True,
    rel_time_s=0.0,
    use_actual_video=True,
    max_examples_per_region=3,
)


## Export GIFs

These cells save example clips while also displaying them inline.

In [ ]:
DOWNLOAD_DIR = ANALYSIS_DIR / 'Downloads'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

example_onset_videos = make_event_example_videos(
    event_kind='syllable_onset',
    label=1,
    social_only=None,
    one_per_region=True,
    n_examples=2,
    window_s=1.5,
    save_dir=DOWNLOAD_DIR,
    use_actual_video=True,
)
example_onset_videos


In [ ]:
example_transition_videos = make_event_example_videos(
    event_kind='transition_onset',
    label='13->5',
    social_only=True,
    one_per_region=True,
    n_examples=2,
    window_s=1.5,
    save_dir=DOWNLOAD_DIR,
    use_actual_video=True,
)
example_transition_videos


## Handy Notes

- `social_only=True` filters to events with social proximity at the event.
- `one_per_region=True` is useful when you want a clean `NAc` vs `mPFC` comparison.
- `one_per_bout_phase=True` is useful when you want one example from each interaction epoch.
- If no matching video is found on disk, the notebook falls back to a pose-only rendering.
